In [ ]:
import pymysql
import pandas as pd
import openai
import requests
import re
import time
import traceback
import os
import openai
from openai import OpenAI


## 1.4 Task-Tree Structure

In [ ]:
class TaskTreeNode:
    def __init__(self, data):
        self.data = data
        self.children = []

    def add_child(self, child):
        self.children.append(child)
        
    def get_subtree(self, data):
        if self.data == data:
            return self

        for child in self.children:
            subtree = child.get_subtree(data)
            if subtree:
                return subtree
        return None
    
    def get_child_node(self, index):
        if 0 <= index < len(self.children):
            return self.children[index]
        else:
            return None
        
    def is_leaf(self):
        return len(self.children) == 0
    
    def are_all_leaves_int(self):
        if self.is_leaf():
            return isinstance(self.data, int)
        else:
            for child in self.children:
                if not child.are_all_leaves_int():
                    return False
            return True
        
    def find_non_int_leaves(self, path=[]):
        if self.is_leaf():
            if not isinstance(self.data, int):
                return [(self.data, path)]
        else:
            non_int_leaves = []
            for i, child in enumerate(self.children):
                child_path = path + [i]
                non_int_leaves.extend(child.find_non_int_leaves(child_path))
            return non_int_leaves
        return []

    def find_nodes_by_value(self, target_value, path=[]):
        nodes = []
        if self.data == target_value:
            nodes.append((self, path))
        for i, child in enumerate(self.children):
            child_path = path + [i]
            nodes.extend(child.find_nodes_by_value(target_value, child_path))
        return nodes


In [ ]:
# Create the root node
root = TaskTreeNode("A")

# Create child nodes
child1 = TaskTreeNode(1)
child2 = TaskTreeNode(2)
child3 = TaskTreeNode(3)
child4 = TaskTreeNode("E")
child5 = TaskTreeNode("F")
# Link the child nodes to the root
root.add_child(child1)
root.add_child(child2)
root.add_child(child3)

child2.add_child(child4)
child4.add_child(child5)


In [ ]:
# Call the method on the root node
target_value = "E"  # Example target value
nodes = root.find_nodes_by_value(target_value)

if nodes:
    print("Nodes with value", target_value, "found:")
    for node, path in nodes:
        print("Node:", node.data, "- Path:", path)
else:
    print("No nodes with value", target_value, "found.")


In [ ]:
target_value = "E"  # Example target value
nodes = root.find_nodes_by_value(target_value)


In [ ]:
for node,path in nodes:
    print(i.data)


In [ ]:
target_value = "A"  # Example target value
nodes = root.find_nodes_by_value(target_value)

if nodes:
    for node,path in nodes:
        subtree = root.get_subtree(node.data)
        for child in subtree.children:
            print(child.data)
else:
    print("No nodes with value", target_value, "found.")


In [ ]:
non_int_leaves = root.find_non_int_leaves()

if non_int_leaves:
    print("Non-integer leaf nodes found:")
    for data, path in non_int_leaves:
        print("Data:", data, "- Path:", path)
else:
    print("No non-integer leaf nodes found.")


In [ ]:
subtree = root.get_subtree("A")
if subtree:
    print("Sub-tree:")
    print(subtree.data)  # Prints "C"
    for child in subtree.children:
        print(child.data)  # Prints the data of the children of "C"
else:
    print("Node not found.")


## 2 Agent

In [ ]:
class LlmAgent:
    def __init__(
            self,
            messages=[],
            type='text',
            model_name='gpt-3.5-turbo-1106',
            temperature=0
    ):
        self.messages = messages
        self.client = OpenAI()
        self.type = type
        self.model_name = model_name
        self.temperature = temperature

    def run(self):
        response = self.client.chat.completions.create(
            model=self.model_name,
            messages=self.messages,
            temperature=self.temperature,
            response_format={"type": self.type}
        )
        return response.choices[0].message.content

    def environmental_feedback(self, feedback):
        self.messages.append({
            'role': 'system',
            'content': feedback
        })

    def continue_conversation(self, user_input):
        self.messages.append({
            'role': 'user',
            'content': user_input
        })
        assistant_response = self.run()
        self.messages.append({
            'role': 'assistant',
            'content': assistant_response
        })

        return assistant_response


In [ ]:
# 简单例子：
# goal = "I want to know the highest spot price of gold in China."
# goal = "I want to know the actual gdp of China after year 2000."
# goal = "I want to see when did Germany has higher actual gdp than China."
# 中级例子
# goal = "I want to see the years when the actual gdp of China are higher than that of the name gdp."
# goal = "I want to know the stock market index of China when China’s house price index reached its highest pitch."
# 困难例子
goal = "I want to see the spot London prices of all metals on 2012-12-12."
#根据kg获取数据库信息
# print("获取任务相关数据库信息")
database_info_message = get_kg_info(goal,table_building_messages)

signal,sub_tasks = Task_Decomposition(goal,str(database_info_message))


In [ ]:
root = TaskTreeNode(goal)
if len(sub_tasks) > 1:
    for i in sub_tasks:
        child = TaskTreeNode(str(i))
        root.add_child(child)


In [ ]:
for i in sub_tasks:
    while signal == "Decomposite":
        database_info_message = get_kg_info(i,table_building_messages)
        signal,sub_tasks = Task_Decomposition(i,str(database_info_message))
        if len(sub_tasks) > 1:
            for x in sub_tasks:
                child = TaskTreeNode(str(x))
                root.add_child(child)


In [ ]:
x = [1,1,1]
si = [0,0,1]

sisi = 0
for i in range(len(x)):
    while sisi == 0:
        sisi = si[i]
    if sisi == 0:
        continue
    x[i] = x[i] +1


In [ ]:
x


In [ ]:
while signal == "Decomposite":
    for i in sub_tasks:
        database_info_message = get_kg_info(i,table_building_messages)
        signal,sub_sub_tasks = Task_Decomposition(i,str(database_info_message))
        if signal == "Not Decomposite":
            continue
        else:


In [ ]:
if len(sub_tasks) > 1:
    for i in sub_tasks:
        database_info_message = get_kg_info(i,table_building_messages)
        signal,sub_sub_tasks = Task_Decomposition(i,str(database_info_message))


In [ ]:
sub_tasks


In [ ]:
subtree = root.get_subtree(goal)

if subtree:
    print("Sub-tree:")
    print(subtree.data)  # Prints "C"
    for child in subtree.children:
        print(child.data)  # Prints the data of the children of "C"
else:
    print("Node not found.")


In [ ]:
child_b = root.get_subtree(goal)


In [ ]:
class TaskTreeNode:
    def __init__(self, data):
        self.data = data
        self.children = []

    def add_child(self, child):
        self.children.append(child)
        
    def is_leaf(self):
        return len(self.children) == 0
        
    def get_subtree(self, data):
        if self.data == data:
            return self

        for child in self.children:
            subtree = child.get_subtree(data)
            if subtree:
                return subtree
        return None
    
    def get_child_node(self, index):
        if 0 <= index < len(self.children):
            return self.children[index]
        else:
            return None


In [ ]:
child_b = root.get_subtree(goal)
if child_b:
    second_child = child_b.get_child_node(3)  # Index 1 represents the second child node
    if second_child:
        print("Second child of B:", second_child.data)
    else:
        print("Second child of B not found.")
else:
    print("Node B not found.")


In [ ]:
for i in range(len(sub_tasks)):
    database_info_message = get_kg_info(sub_tasks[i],table_building_messages)
    signal,sub_sub_tasks = Task_Decomposition(sub_tasks[i],str(database_info_message))
    if len(sub_sub_tasks) != 1:
        sub_tasks


In [ ]:
sub_tasks


In [ ]:
# Create the root node
root = TaskTreeNode("A")

# Create child nodes
child1 = TreeNode(str("BBBBB"))
child2 = TreeNode("C")
child3 = TreeNode("D")
child4 = TreeNode("E")
child5 = TreeNode("F")
# Link the child nodes to the root
root.add_child(child1)
root.add_child(child2)
root.add_child(child3)

child2.add_child(child4)
child4.add_child(child5)


In [ ]:
root = TaskTreeNode("A")
x = [1,2,3,4,5]
for i in range(5):
    child1 = TreeNode(x[i])
    root.add_child(child1)


In [ ]:
subtree = root.get_subtree("A")


In [ ]:
subtree


In [ ]:
subtree = root.get_subtree("A")
if subtree:
    print("Sub-tree:")
    print(subtree.data)  # Prints "C"
    for child in subtree.children:
        print(child.data)  # Prints the data of the children of "C"
else:
    print("Node not found.")


In [ ]:
x = ['a','b','c']
y = ['1','2','3']


In [ ]:
x[0].append(y)


In [ ]:
print(x)
print(y)


In [ ]:
y


In [ ]:
messages = [{"role":"system","content":"You are a precise, efficient and concise SQL engineer."}]
TaskDecompose_agent = LlmAgent(messages=messages)
TaskDecompose_agent.environmental_feedback(TaskDecompose_prompt)
response = TaskDecompose_agent.continue_conversation(user_input = "let us begin:" + goal + str(database_info_message))


In [ ]:
print(response)


In [ ]:
match = re.search(r"\[Decomposite or not\]:\s*(.*)", response)


In [ ]:
decomposite_or_not = match.group(1)
if 'yes' in decomposite_or_not:
    print("yes")
